# 20 · Human confirmation for refunds

ADK FunctionTool can pause for confirmation before executing an action. This demo records only an in-session simulated refund; it never contacts a payment system.

**Runtime:** ADK 2.10.0 on `aadhi`, existing `dataengenv` kernel and ADC.

**Cost:** Lite model calls. Synthetic ecommerce data only. Run cells from top to bottom; each notebook creates its own sessions.


In [ ]:
from pathlib import Path
import sys
# Works when Jupyter starts in GoogleADK or its parent folder.
root = Path.cwd() if (Path.cwd() / "course.py").exists() else Path.cwd() / "GoogleADK"
if not (root / "course.py").exists():
    raise RuntimeError("Open this notebook with GoogleADK as the working directory.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from course import (ROOT, CONFIG, MODEL, PROJECT, LOCATION, SKIP_SSL, DATA,
                    llm, gemini, make_lab, ask, final_text, token_usage,
                    get_order, get_product, get_customer, types)
print(f"Project={PROJECT} | location={LOCATION} | model={MODEL} | TLS bypass={SKIP_SSL}")


In [ ]:
from google.adk.tools import FunctionTool, ToolContext

def record_demo_refund(order_id: str, tool_context: ToolContext) -> dict:
    """Record a simulated full refund for a known demo order; no payment is sent."""
    order = get_order(order_id)
    if "error" in order:
        return order
    key = "refund:" + order_id
    if tool_context.state.get(key):
        return {"status": "already_recorded", "order_id": order_id}
    tool_context.state[key] = {"amount": order["total"], "currency": order["currency"]}
    return {"status": "simulated_refund_recorded", "order_id": order_id}

tool = FunctionTool(record_demo_refund, require_confirmation=True)
lab = await make_lab(llm("refund_assistant", "Use record_demo_refund when requested. Respect a denied confirmation.", tools=[tool]))
events = await ask(lab, "Record a demo full refund for O1001.")
pending = [call for e in events for call in e.get_function_calls() if call.name == "adk_request_confirmation"]
assert pending, "Expected an ADK confirmation request before the action."
print("Confirmation request:", pending[0].args)
assert "refund:O1001" not in lab.session.state


In [ ]:
# Default: deny. Change this explicit classroom decision to True to simulate approval.
APPROVE_DEMO_REFUND = False
request = pending[0]
await ask(lab, parts=[types.Part(function_response=types.FunctionResponse(
    id=request.id, name=request.name, response={"confirmed": APPROVE_DEMO_REFUND}))])
print("Refund state:", lab.session.state.get("refund:O1001", "No refund recorded"))


## Try it

Run once denied and once approved in separate fresh sessions. In an application the authenticated human UI, not the model, supplies the confirmation.


## Reference

[Official ADK documentation](https://adk.dev/tools-custom/confirmation/). Shared configuration: `config.json`. No environment activation or login cells are needed.
